In [51]:
import pandas as pd 
import numpy as np

In [52]:
df_petrol= pd.read_csv("data-petrol.csv")
df_ev = pd.read_csv("ev-charging-stations-india.csv")
print("Files successfully load ho gayi")
print(f" total petrol pumps : {len(df_petrol)}")
print(f" total existing ev stations: {len(df_ev)}")


Files successfully load ho gayi
 total petrol pumps : 2217
 total existing ev stations: 1547


In [53]:
df_petrol.columns


Index(['PumpID', 'Pump Name', 'Brand', 'Review', 'Rating', 'City', 'State',
       'Latitude', 'Longitude'],
      dtype='object')

In [54]:
df_ev.columns

Index(['name', 'state', 'city', 'address', 'lattitude', 'longitude', 'type'], dtype='object')

In [55]:

df_petrol_clean = df_petrol[['PumpID', 'Pump Name', 'Brand', 'City', 'State', 'Latitude', 'Longitude']].copy()
df_petrol_clean.columns = ['id', 'name', 'brand', 'city', 'state', 'latitude', 'longitude']

df_ev_clean = df_ev[['name', 'city', 'state', 'address', 'lattitude', 'longitude', 'type']].copy()
df_ev_clean.columns = ['name', 'city', 'state', 'address', 'latitude', 'longitude', 'charger_type']


print("Petrol columns:", df_petrol_clean.columns.tolist())
print("EV columns:", df_ev_clean.columns.tolist())

Petrol columns: ['id', 'name', 'brand', 'city', 'state', 'latitude', 'longitude']
EV columns: ['name', 'city', 'state', 'address', 'latitude', 'longitude', 'charger_type']


In [56]:

df_petrol_clean['latitude'] = pd.to_numeric(df_petrol_clean['latitude'], errors='coerce')
df_petrol_clean['longitude'] = pd.to_numeric(df_petrol_clean['longitude'], errors='coerce')

df_ev_clean['latitude'] = pd.to_numeric(df_ev_clean['latitude'], errors='coerce')
df_ev_clean['longitude'] = pd.to_numeric(df_ev_clean['longitude'], errors='coerce')


df_petrol_clean = df_petrol_clean.dropna(subset=['latitude', 'longitude'])
df_ev_clean = df_ev_clean.dropna(subset=['latitude', 'longitude'])

# 3. Valid India Latitude & Longitude Filter (Lat: 8 to 38, Long: 68 to 98)
# Isse koi galat/out-of-range coordinate hoga toh clean ho jayega
df_petrol_clean = df_petrol_clean[
    (df_petrol_clean['latitude'].between(8, 38)) & 
    (df_petrol_clean['longitude'].between(68, 98))
]

df_ev_clean = df_ev_clean[
    (df_ev_clean['latitude'].between(8, 38)) & 
    (df_ev_clean['longitude'].between(68, 98))
]

print(f" Valid Petrol Pumps: {len(df_petrol_clean)}")
print(f" Valid EV Stations: {len(df_ev_clean)}")

 Valid Petrol Pumps: 2210
 Valid EV Stations: 1533


In [57]:
# clean files save 
df_petrol_clean.to_csv("cleaned_petrol_pumps.csv", index=False)
df_ev_clean.to_csv("cleaned_ev_stations.csv", index= False)

print("- petrol pumpd sample - ")
display(df_petrol_clean.head())

print("\n - ev stations sample - ")
display(df_ev_clean.head())

- petrol pumpd sample - 


,id,name,brand,city,state,latitude,longitude
0,AP001,Sai Fuel Station,Indian Oil,Visakhapatnam,Andhra Pradesh,17.6868,83.2185
1,AP002,RK Petrol Pump,HP,Visakhapatnam,Andhra Pradesh,17.6880,83.2210
2,AP003,VSP Fuel Hub,Bharat Petroleum,Visakhapatnam,Andhra Pradesh,17.6895,83.2150
3,AP004,Galaxy Fuel,Reliance,Visakhapatnam,Andhra Pradesh,17.6872,83.2170
4,AP005,Star Petrol,Indian Oil,Visakhapatnam,Andhra Pradesh,17.6850,83.2205



 - ev stations sample - 


,name,city,state,address,latitude,longitude,charger_type
0,Neelkanth Star DC Charging Station,Gurugram,Haryana,"Neelkanth Star Karnal, NH 44, Gharunda, Kutail...",29.6019,76.9803,12.0
1,Galleria DC Charging Station,Gurugram,Haryana,"DLF Phase IV, Sector 28, Gurugram, Haryana 122022",28.4673,77.0818,12.0
2,Highway Xpress (Jaipur-Delhi) DC charging station,Behror,Rajasthan,"Jaipur to Delhi Road, Behror Midway, Behror, R...",27.8751,76.2760,12.0
3,Food Carnival DC Charging Station,Khatauli,Uttar Pradesh,"Fun and Food Carnival, NH 58, Khatauli Bypass,...",29.3105,77.7218,12.0
4,Food Carnival AC Charging Station,Khatauli,Uttar Pradesh,"NH 58, Khatauli Bypass, Bhainsi, Uttar Pradesh...",29.3105,77.7218,12.0


In [58]:
from sklearn.cluster import KMeans


X = df_petrol_clean[['latitude', 'longitude']]


k = 50


kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(X)


centroids = kmeans.cluster_centers_


recommended_stations = pd.DataFrame(centroids, columns=['latitude', 'longitude'])
recommended_stations['Proposed_Station_ID'] = [f'PROP_EV_{i+1:02d}' for i in range(k)]
recommended_stations['Priority'] = 'High Demand Cluster'

# Save to CSV
recommended_stations.to_csv("recommended_ev_locations.csv", index=False)

print(" 50 Best Optimal Charging Locations Calculate Ho Gaye!")
display(recommended_stations)

 50 Best Optimal Charging Locations Calculate Ho Gaye!


,latitude,longitude,Proposed_Station_ID,Priority
0,21.529780,72.825513,PROP_EV_01,High Demand Cluster
1,25.506924,87.714297,PROP_EV_02,High Demand Cluster
2,28.137265,75.632951,PROP_EV_03,High Demand Cluster
3,11.136911,78.711722,PROP_EV_04,High Demand Cluster
4,26.875085,94.621537,PROP_EV_05,High Demand Cluster
5,19.641452,79.174560,PROP_EV_06,High Demand Cluster
6,31.556300,75.521089,PROP_EV_07,High Demand Cluster
7,15.423154,74.049204,PROP_EV_08,High Demand Cluster
8,13.095229,79.345504,PROP_EV_09,High Demand Cluster
9,16.771416,81.327838,PROP_EV_10,High Demand Cluster


In [59]:
%pip install folium
import folium
print("Folium successfully import ho gaya!")

Note: you may need to restart the kernel to use updated packages.
Folium successfully import ho gaya!


In [60]:




df_ev = pd.read_csv("cleaned_ev_stations.csv")
df_recommended = pd.read_csv("recommended_ev_locations.csv")

print("Files loaded successfully!")

Files loaded successfully!


In [61]:

map_center = [df_recommended['latitude'].mean(), df_recommended['longitude'].mean()]

# 2. Base Map initialize karo
ev_map = folium.Map(location=map_center, zoom_start=10, tiles='CartoDB positron')

# 3. Existing EV Stations plot karo (Blue Circles)
for _, row in df_ev.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='#1f77b4',
        fill=True,
        fill_color='#1f77b4',
        fill_opacity=0.6,
        tooltip=f"Existing EV: {row.get('name', 'EV Station')}"
    ).add_to(ev_map)

# 4. Proposed Optimal Stations plot karo (Green Icons with Bolt Symbol)
for _, row in df_recommended.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        icon=folium.Icon(color='green', icon='bolt', prefix='fa'),
        tooltip=f"Recommended Spot: {row['Proposed_Station_ID']}",
        popup=f"<b>Station ID:</b> {row['Proposed_Station_ID']}<br><b>Priority:</b> {row['Priority']}"
    ).add_to(ev_map)

# 5. Map ko HTML file me save karo
ev_map.save("ev_network_optimization_map.html")
print(" Map ready aur 'ev_network_optimization_map.html' me save ho gaya!")

# Jupyter screen par direct display karein
ev_map

 Map ready aur 'ev_network_optimization_map.html' me save ho gaya!


In [62]:
import pandas as pd
import numpy as np

# Files load karo
df_p = pd.read_csv("cleaned_petrol_pumps.csv")
df_e = pd.read_csv("cleaned_ev_stations.csv")
df_r = pd.read_csv("recommended_ev_locations.csv")

# 1. Existing EV Stations format
df_e_pbi = pd.DataFrame({
    'Station_ID': [f'EXIST_EV_{i+1:03d}' for i in range(len(df_e))],
    'Name': df_e['name'],
    'City': df_e['city'],
    'State': df_e['state'],
    'Latitude': df_e['latitude'],
    'Longitude': df_e['longitude'],
    'Status': 'Existing EV Station',
    'Category': 'Operational',
    'Estimated_Cost_Lakhs': 0.0,
    'Charger_Type': df_e.get('charger_type', 'Standard DC Fast'),
    'Expected_Daily_Users': np.random.randint(25, 60, len(df_e))
})

# 2. Recommended 50 EV Stations format
np.random.seed(42)
df_r_pbi = pd.DataFrame({
    'Station_ID': df_r['Proposed_Station_ID'],
    'Name': df_r['Proposed_Station_ID'] + ' Hub',
    'City': 'Strategic Zone',
    'State': 'Regional Hub',
    'Latitude': df_r['latitude'],
    'Longitude': df_r['longitude'],
    'Status': 'Proposed Optimal Station',
    'Category': 'Proposed Expansion',
    'Estimated_Cost_Lakhs': np.random.choice([12.5, 18.0, 25.0], len(df_r)), # 50kW / 120kW setup cost
    'Charger_Type': np.random.choice(['Ultra-Fast DC (120kW)', 'Dual Gun Fast DC (60kW)'], len(df_r)),
    'Expected_Daily_Users': np.random.randint(40, 95, len(df_r))
})

# Combine into one Master CSV
df_master = pd.concat([df_e_pbi, df_r_pbi], ignore_index=True)
df_master.to_csv("powerbi_ev_master_data.csv", index=False)
print(" Power BI Ready Dataset: 'powerbi_ev_master_data.csv' created successfully!")

 Power BI Ready Dataset: 'powerbi_ev_master_data.csv' created successfully!
